# Day 50: NVIDIA Dynamo: Disaggregated Prefill Experiment

**Layer:** Implementation


## Concept Overview

Disaggregated serving separates prefill (compute-bound) and decode (memory-bandwidth-bound) into separate worker pools. NVIDIA Dynamo implements this with NIXL for KV cache transfer.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. NVIDIA Dynamo: Disaggregated Prefill Experiment

Run a prefill worker and a decode worker separately, transfer KV cache between them, and measure the overhead vs coupled serving.


In [ ]:
print('NVIDIA Dynamo disaggregated prefill experiment:')
print()
print('Architecture:')
print('  [Client] → [Router] → [Prefill Worker] → KV Transfer → [Decode Worker] → [Client]')
print()

import numpy as np

def simulate_disaggregated(prompt_len, output_len, n_layers=32, d_model=4096,
                            kv_bw_gb_s=900, prefill_tflops=312, decode_bw_gb_s=2000):
    # Prefill time
    prefill_flops = 2 * prompt_len**2 * d_model * n_layers
    t_prefill_ms = prefill_flops / (prefill_tflops * 1e12) * 1000
    # KV transfer time
    kv_bytes = 2 * n_layers * prompt_len * d_model * 2  # K+V, FP16
    t_kv_ms = kv_bytes / (kv_bw_gb_s * 1e9) * 1000
    # Decode time (memory-bound)
    weight_bytes = n_layers * 4 * d_model**2 * 2  # per token
    t_decode_per_token_ms = weight_bytes / (decode_bw_gb_s * 1e9) * 1000
    t_decode_ms = output_len * t_decode_per_token_ms
    ttft = t_prefill_ms + t_kv_ms
    return {'ttft_ms': ttft, 't_prefill': t_prefill_ms, 't_kv': t_kv_ms,
            't_decode': t_decode_ms, 'total': ttft + t_decode_ms}

print(f'{"Prompt":>8} {"Output":>8} {"Prefill":>10} {"KV Xfer":>10} {"TTFT":>8} {"Total":>8}')
for prompt in [128, 512, 2048, 8192]:
    r = simulate_disaggregated(prompt, 256)
    print(f'{prompt:>8} {256:>8} {r["t_prefill"]:>9.1f}ms {r["t_kv"]:>9.2f}ms {r["ttft_ms"]:>7.1f}ms {r["total"]:>7.1f}ms')


## 2. Break-Even Analysis

At what prompt length does disaggregation break even — when does KV transfer overhead become worthwhile?


In [ ]:
print('Break-even analysis: when is disaggregation worth it?')
print()
prompt_lens = [64, 128, 256, 512, 1024, 2048, 4096]
for prompt in prompt_lens:
    # Coupled: prefill and decode on same GPU
    r_coupled = simulate_disaggregated(prompt, 256, kv_bw_gb_s=1e9)  # no KV transfer
    # Disaggregated: separate workers
    r_disagg = simulate_disaggregated(prompt, 256)
    benefit = r_coupled['total'] - r_disagg['total']
    print(f'  prompt={prompt:>5}: coupled={r_coupled["total"]:>7.1f}ms disagg={r_disagg["total"]:>7.1f}ms benefit={benefit:>+7.1f}ms')


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- Disaggregated serving separates prefill (compute-bound) and decode (memory-bandwidth-bound) into separate worker pools.
- Disaggregation is most beneficial for long prompts: at seq=8192, prefill takes 200ms while KV transfer via NVLink takes <1ms — a clear win. At seq=128, the 10ms KV transfer overhead makes it marginal..
- Day 50 implementation complete.
